# Document Intelligence Workshop: Statements of Work with Genie and Agent Bricks

Northgate's commercial terms (fees, fee caps, agreed day rates, payment terms) only exist in signed Statement of Work PDFs. Part 1 gets answers from those documents with the Genie Agent from [1. Genie Agents](#notebook-3452153026155247), first by letting Genie read the files directly, then by querying terms already extracted into tables. Part 2 uses the Agent Bricks Document Parsing and Information Extraction tools to build, test and evaluate the `ai_parse_document` and `ai_extract` logic without writing it by hand.

## Two ways to get answers from documents

| | A. Genie reads the files | B. Extract to tables, then Genie queries them |
| --- | --- | --- |
| How | Attach a volume of PDFs to the Genie Agent; Agent mode retrieves the relevant pages | `ai_parse_document` + `ai_extract` write the terms to Delta tables; Genie writes SQL over them |
| Good for | Open questions, one-off lookups, wording (*"what does the SOW say about expenses?"*) | Precise, repeatable numbers across every contract (*"which engagements are over their fee?"*) |
| Answers | Text with citations to the page of the PDF | SQL + result table, benchmarkable |
| Set-up effort | Minutes | An extraction step to build, test and schedule |

Most real solutions use both. This notebook uses both, then shows how Agent Bricks helps you build option B faster.

| Part | Topic | Format |
| --- | --- | --- |
| 0 | Configuration and a clean contracts volume | Run cells |
| 1A | Genie Agent reads the PDFs directly (Agent mode) | Hands-on |
| 1B | Add the extracted contract tables to the Genie Agent | Hands-on |
| 2A | Agent Bricks Document Parsing: see what `ai_parse_document` sees | Hands-on |
| 2B | Agent Bricks Information Extraction: build, evaluate and export `ai_extract` | Hands-on |

**Prerequisites:** the [1. Workshop Data Setup](#notebook-3120290690157533) notebook has been run, and you have built the Genie Agent in [1. Genie Agents](#notebook-3452153026155247).

## Part 0: Configuration and a clean contracts volume

The `raw_files` volume also holds timesheets, expenses and the facilitator's answer key. If you attach that volume to Genie, it may read files it shouldn't (including the answers). The next cell copies only the contract PDFs into a dedicated `contracts` volume. **Only give an agent the documents it needs.**

In [0]:
dbutils.widgets.text("catalog", "praju_stable_2_catalog", "Catalog")
dbutils.widgets.text("schema", "consulting_billing", "Schema")
dbutils.widgets.text("volume", "raw_files", "Volume")

CATALOG = dbutils.widgets.get("catalog")
SCHEMA = dbutils.widgets.get("schema")
VOLUME = dbutils.widgets.get("volume")

FQ = f"`{CATALOG}`.`{SCHEMA}`"
VOLUME_PATH = f"/Volumes/{CATALOG}/{SCHEMA}/{VOLUME}"
CONTRACTS_PATH = f"/Volumes/{CATALOG}/{SCHEMA}/contracts"
spark.sql(f"USE {FQ}")

if not spark.catalog.tableExists(f"{CATALOG}.{SCHEMA}.timesheet_entries"):
    raise RuntimeError("timesheet_entries is missing: run Parts 0-1 of the '1. Genie Agents' notebook first.")

# A volume that contains only the signed contract documents
spark.sql(f"CREATE VOLUME IF NOT EXISTS {FQ}.contracts COMMENT 'Signed Statements of Work (SOW-26-NNN.pdf) and change requests (CR-26-NNN-NN.pdf) between Northgate Advisory and its clients. One PDF per document.'")
for f in dbutils.fs.ls(f"{VOLUME_PATH}/contracts"):
    if f.name.endswith(".pdf"):
        dbutils.fs.cp(f.path, f"{CONTRACTS_PATH}/{f.name}")

files = sorted(f.name for f in dbutils.fs.ls(CONTRACTS_PATH))
print(f"{len(files)} documents in {CONTRACTS_PATH}: {', '.join(files)}")

## Part 1A: Let the Genie Agent read the PDFs (hands-on)

File analysis is a Public Preview feature; a workspace admin enables it on the **Previews** page. It runs in **Agent mode** only.

1. Open your Genie Agent from [1. Genie Agents](#notebook-3452153026155247) and go to **Configure > Data**.
2. Click **Add** and choose the volume `<catalog>.consulting_billing.contracts` (not `raw_files`).
3. Give it a description. Genie uses it to decide when to look in the files:
   *"Signed Statements of Work and change requests for each client engagement. Each SOW states the billing basis, fee or fee cap, agreed day rate per grade, payment terms, client PO, expense policy and signatories. File names are the SOW reference, e.g. SOW-26-003.pdf. A change request (CR-...) amends the fee of an existing SOW."*
4. Switch the chat to **Agent mode** and ask:

| # | Question | What a correct answer contains |
| --- | --- | --- |
| 1 | *What are the payment terms and PO number in the SOW for ENG-26-003?* | PO-413898, 30 days, with a citation to the page in SOW-26-003.pdf |
| 2 | *Which SOWs say that expenses are included in the fees?* | SOW-26-001, 003, 005, 011, 012 and 013 |
| 3 | *What did change request CR-26-005-01 change, and why?* | Fee cap up by GBP 45,000 to GBP 257,000; two extra business services in scope |
| 4 | *Who signed the SOW for the Thistle Insurance engagement on behalf of the client?* | A name and job title from the signature block on the last page |
| 5 | *Is ENG-26-003 over its fee cap? Use the SOW for the cap and the timesheets for the work done.* | Combines the PDF with `timesheet_entries`: yes, over the GBP 128,000 cap |

Click the citations and check they point to the right page. Question 5 is the interesting one: Genie mixes a document with structured data in one answer.

**Discussion:** try asking question 5 for *all* 16 engagements. It's slow, and you can't benchmark it. That's the signal to move to option B.

## Part 1B: Add the extracted contract tables to your Genie Agent (hands-on)

The facilitator has already extracted the terms from every PDF into tables with `ai_parse_document` and `ai_extract` (the same AI functions as the [3. Extension - PDF Parsing](#notebook-102205435070834) demo): `contract_documents`, `contract_terms`, `contracted_rates` and `engagement_fee_burn`. AI output is stored **once** in tables, so Genie answers from the same values every time.

1. In your Genie Agent, **Configure > Data > Add**: `contract_terms`, `contracted_rates`, `engagement_fee_burn`. Leave out `contract_documents` (it holds large parsed documents that Genie doesn't need for SQL).
2. Add this to the **text instructions**:
```
- Contract terms come from the signed PDFs (contract_terms, contracted_rates). When an engagement has a Change Request,
  its fee_amount_gbp replaces the SOW fee. engagement_fee_burn.current_fee_gbp already applies this rule.
- "Fee burn" = WIP at contracted rates / current fee. Use engagement_fee_burn for fee burn, realisation and over-fee questions.
- "Rate realisation" = WIP at contracted (agreed) rates / WIP at standard rates.
```
3. Add joins: `contract_terms.engagement_code = engagements.engagement_code` and `contracted_rates.engagement_code = engagements.engagement_code`.
4. Ask in **Chat mode**, then add each one as a **benchmark**:

| # | Question | Expected answer |
| --- | --- | --- |
| 1 | *Which engagements are over their fee?* | ENG-26-003, ENG-26-007, ENG-26-010 |
| 2 | *What is the current fee cap for ENG-26-005, and has it changed?* | GBP 257,000, up from GBP 212,000 by change request CR-26-005-01 |
| 3 | *What is the average rate realisation by practice?* | Uses `engagement_fee_burn` joined to `engagements` |
| 4 | *Which SOWs have expenses included in fees?* | The same six as Part 1A: SOW-26-001, 003, 005, 011, 012, 013 |
| 5 | *What are the agreed day rates for a Manager across all engagements?* | From `contracted_rates`; shows discounts by client |
| 6 | *How many hours were booked by grades that the SOW doesn't price?* | 4 hours: 2 on ENG-26-001 and 2 on ENG-26-002 (`hours_without_contract_rate`) |

Now re-ask the Part 1A question 5 (*"Is ENG-26-003 over its fee cap?"*) in Chat mode. Compare the speed, the answer and whether you could benchmark it.

## Part 2: Build the extraction with Agent Bricks

Writing the `ai_extract` schema by hand works, but it's slow to get right and hard to test. Agent Bricks gives you two UI tools that do the same job, then hand you the SQL.

### 2A. Document Parsing: see what `ai_parse_document` sees (hands-on, 5 min)
1. Click **Agents** in the left navigation, then **Create Agent > Document Parsing**.
2. Select a document from Unity Catalog: `<catalog>.consulting_billing.contracts`, file `SOW-26-005.pdf`. Click **Parse document**.
3. The source PDF appears on the left and the parsed result on the right. Switch between **Formatted text** and **Raw JSON**:
   * Is the agreed day rate table kept as a table (HTML), with every grade in the right row?
   * Are the signature blocks on the last page picked up as text?
   * Find the element `type` values in the JSON (e.g. `section_header`, `text`, `table`). These are what `ai_extract` reads.
4. Try `CR-26-005-01.pdf` too. It's a different layout; does the parser still find the fee lines?
5. Click **Use Agent > Notebook**. This is the same `ai_parse_document` function that built `contract_documents`.

### 2B. Information Extraction: build, evaluate and export `ai_extract` (hands-on, 20 min)

**Step 1. Data:** **Agents > Create Agent > Information Extraction**. Click **Select volume** and choose `<catalog>.consulting_billing.contracts`. Click **Create Agent**.

**Step 2. Schema:** under **Configuration**, describe what you want and click **Generate Schema**:
> *From each Statement of Work extract: the engagement code (ENG-26-NNN), the fee or fee cap in GBP, the client purchase order number, the payment terms in days, and the agreed day rate table as a list of grade and agreed day rate in GBP.*

Review the generated fields. For the evaluation in Step 4 to line up, rename them to exactly: `engagement_code`, `fee_amount_gbp`, `purchase_order`, `payment_terms_days`, `day_rates` (items: `grade`, `agreed_day_rate_gbp`). Click **Save and run extraction**.

**Step 3. Improve:** open the options next to **Save and run extraction** and turn on **Citations** and **Confidence scores**. For any document where a value is wrong or has low confidence, type feedback in plain English (for example *"the fee is in section 4, not the day-rate total"*) and run again. Use **Versions > Compare** to see how the schema changed.

**Step 4. Evaluate:** the facilitator has built a labelled evaluation table, `sow_extraction_eval`, from the answer key. In the workbench click **Run evaluation**, choose that table, map **Input** to `parsed` and **Ground truth** to `ground_truth`, and run. Review **Overall field accuracy**, the **Per-field scores** and any failed documents. Try **Fix with Genie** on any finding.

**Step 5. Use it:** click **Use Agent > Run in SQL**. The generated query uses `ai_extract` with your tuned schema. You could also choose **Create a Lakeflow pipeline** so new SOWs are extracted as they land in the volume.

**Discussion (Audit):** the evaluation run is your evidence that the extraction is accurate enough to rely on, field by field. It's a documented, re-runnable test, the same idea as the Genie benchmarks in [1. Genie Agents](#notebook-3452153026155247).

## Wrap-up

| You want to... | Use |
| --- | --- |
| Look something up in a few documents, with citations | Genie Agent with an attached volume (Agent mode) |
| Report precise numbers across every contract | `ai_parse_document` + `ai_extract` into tables, then Genie in Chat mode |
| Build and tune the extraction schema without hand-writing JSON | Agent Bricks Information Extraction |
| Prove the extraction is accurate | Information Extraction evaluation with a labelled dataset |
| See why a document extracts badly | Agent Bricks Document Parsing |

**Next:** [3. Agents](#notebook-3452153026155249) uses this Genie Agent as a tool in an AI Playground agent that drafts new SOWs.

### Facilitator answer key
* 17 documents parse with no errors; 16 are Statements of Work and 1 is a Change Request.
* Extraction accuracy against the answer key: 16/16 fees, POs and payment terms; 79/79 day rates (re-check after every re-extraction).
* Over fee: ENG-26-003 (about 109%), ENG-26-007 (about 109%), ENG-26-010 (about 101%).
* ENG-26-005: SOW fee cap GBP 212,000; change request CR-26-005-01 raises it by GBP 45,000 to GBP 257,000.
* Expense policy: included in fees for SOW-26-001, 003, 005, 011, 012, 013; recharged at cost for 002, 004, 006, 009; recharged at cost capped at 8% of fees for 007, 008, 010, 014, 015, 016.
* ENG-26-003 terms: PO-413898, 30 days.
* Hours with no contracted rate: 4 (2 on ENG-26-001, 2 on ENG-26-002).
* `sow_extraction_eval` has 16 rows, one per SOW.